# Module 4 · Prompt Engineering
Zero-shot, few-shot, and chain-of-thought prompting — with side-by-side comparisons.

---
> **Setup:** Run the first two cells before anything else.
> API key is loaded automatically from the `.env` file in the parent folder.

In [1]:
%pip install -q google-genai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os, json, math, time, base64, getpass, re, urllib.request
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()  # picks up modules/.env symlink
api_key = os.environ.get('GEMINI_API_KEY')
if not api_key:
    api_key = getpass.getpass('Paste your Gemini API key: ')

client = genai.Client(api_key=api_key)
MODEL = 'gemma-4-26b-a4b-it'
DEFAULT_MAX = 2048

def cfg(**kwargs):
    kwargs.setdefault('max_output_tokens', DEFAULT_MAX)
    kwargs.setdefault('temperature', 0.7)
    return types.GenerateContentConfig(**kwargs)

def get_text(response) -> str:
    if response.text:
        return response.text.strip()
    parts = []
    for cand in (response.candidates or []):
        if cand.content:
            for part in cand.content.parts:
                if not getattr(part, 'thought', False) and part.text:
                    parts.append(part.text)
    return ''.join(parts).strip()

def _call_with_retry(fn, *args, max_retries=5, **kwargs):
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            msg = str(e)
            if '429' in msg or 'RESOURCE_EXHAUSTED' in msg:
                # parse suggested retry delay from error (e.g. 'retryDelay': '30s')
                m = re.search(r'retry[^0-9]*([0-9]+)s', msg, re.I)
                wait = int(m.group(1)) + 5 if m else 35
                print(f'  ⏳ Rate-limited — waiting {wait}s (attempt {attempt+1}/{max_retries})')
                time.sleep(wait)
            else:
                raise
    raise RuntimeError('Max retries exceeded')

_raw_gen    = client.models.generate_content
_raw_stream = client.models.generate_content_stream
_raw_embed  = client.models.embed_content
_raw_count  = client.models.count_tokens
client.models.generate_content        = lambda *a,**kw: _call_with_retry(_raw_gen,    *a,**kw)
client.models.generate_content_stream = lambda *a,**kw: _call_with_retry(_raw_stream, *a,**kw)
client.models.embed_content           = lambda *a,**kw: _call_with_retry(_raw_embed,  *a,**kw)
client.models.count_tokens            = lambda *a,**kw: _call_with_retry(_raw_count,  *a,**kw)

print('\u2705 Setup complete. Model:', MODEL, '| Retry-on-429: enabled')


✅ Setup complete. Model: gemma-4-31b-it | Retry-on-429: enabled


### 🔌 API Key Test

In [ ]:
try:
    _r = client.models.generate_content(
        model=MODEL,
        contents="Reply with exactly the words: Hello LLM course",
        config=cfg(temperature=0.0)
    )
    print("✅ API key working!")
    print("Model says:", get_text(_r))
except Exception as e:
    print("❌ API key error:", e)

✅ API key working!
Model says: Hello LLM course


---
## 7. Prompt Engineering Patterns

| Pattern | What it is | When to use |
|---|---|---|
| **Zero-shot** | Just ask, no examples | Simple well-known tasks |
| **Few-shot** | Include 2–5 examples | Specific format or rare classification |
| **Chain-of-thought** | Ask to reason step-by-step | Math, logic, multi-step problems |

In [ ]:
# Zero-shot — just ask
r = client.models.generate_content(
    model=MODEL,
    contents="Classify as POSITIVE, NEGATIVE, or NEUTRAL (one word only):\nReview: 'The battery life is mediocre but the camera is stunning.'",
    config=cfg(temperature=0.0)
)
print("Zero-shot:", get_text(r))

Zero-shot: POSITIVE


In [5]:
# Few-shot — examples teach the exact output format
few_shot_prompt = """Classify sentiment. Reply with ONLY one word: POSITIVE, NEGATIVE, or NEUTRAL.

Review: "This laptop is blazing fast and the build quality is excellent."
Sentiment: POSITIVE

Review: "Terrible customer service. Waited 3 weeks and got the wrong item."
Sentiment: NEGATIVE

Review: "It arrived on time. Does what it says on the box."
Sentiment: NEUTRAL

Review: "The battery life is mediocre but the camera is stunning."
Sentiment:"""

r_few = client.models.generate_content(
    model=MODEL, contents=few_shot_prompt,
    config=cfg(temperature=0.0, stop_sequences=["\n"])
)
print("Few-shot:", get_text(r_few))

Few-shot: NEUTRAL


In [ ]:
# Chain-of-thought — compare direct vs step-by-step
hard_question = (
    "A store has 3 shelves. Each shelf has 4 rows. Each row has 8 items. "
    "On Monday, 20% of items are sold. On Tuesday, 15 more items are added. "
    "How many items are in the store at the end of Tuesday?"
)

r_direct = client.models.generate_content(
    model=MODEL,
    contents=hard_question + "\nAnswer with ONLY a number, nothing else.",
    config=cfg(temperature=0.0)
)
print("Direct (no CoT):", get_text(r_direct))

r_cot = client.models.generate_content(
    model=MODEL,
    contents=hard_question + "\nThink step by step, then give the final answer.",
    config=cfg(temperature=0.0, max_output_tokens=4096)
)
print("\nChain-of-thought:")
print(get_text(r_cot))

# Manual verification
total = 3 * 4 * 8
after_monday = int(total * 0.80)
after_tuesday = after_monday + 15
print(f"\n✅ Manual check: {total} → {after_monday} → {after_tuesday}")

Direct (no CoT): 



Chain-of-thought:
To find the total number of items in the store at the end of Tuesday, we will follow these steps:

1.  **Calculate the initial number of items:**
    The store has 3 shelves, each with 4 rows, and each row has 8 items.
    Total items = 3 shelves × 4 rows/shelf × 8 items/row
    Total items = 12 rows × 8 items/row = **96 items**

2.  **Calculate the items sold on Monday:**
    On Monday, 20% of the items were sold.
    Items sold = 20% of 96
    Items sold = 0.20 × 96 = **19.2 items**

3.  **Calculate the items remaining after Monday:**
    Remaining items = Initial items - Items sold
    Remaining items = 96 - 19.2 = **76.8 items**

4.  **Calculate the items at the end of Tuesday:**
    On Tuesday, 15 more items were added.
    Final total = Remaining items + Items added
    Final total = 76.8 + 15 = **91.8 items**

Final Answer: **91.8**

✅ Manual check: 96 → 76 → 91
